In [1]:
!pip install -q -U ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 kB 7.7 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import json
import random
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import ultralytics
import yaml
from collections import Counter
from tqdm.auto import tqdm

from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


In [ ]:
DATASET_ROOT = Path(
    "/content/datasets"
)
 
DATA_YAML_PATH = Path("/content/drive/MyDrive/vision_unit_02_outputs/block_02/crack_seg_local.yaml")

OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/"
    "vision_unit_02_outputs/"
    "block_04"
)

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
RUN_NAME = "yolo26n_crack_seg_v1"
RUN_DIRECTORY = OUTPUT_ROOT / RUN_NAME


BEST_MODEL_PATH = (
    RUN_DIRECTORY
    / "weights"
    / "best.pt"
)

LAST_MODEL_PATH = (
    RUN_DIRECTORY
    / "weights"
    / "last.pt"
)

print("Dataset YAML:", DATA_YAML_PATH)
print("Output:", OUTPUT_ROOT)

Dataset YAML: /content/drive/MyDrive/vision_unit_02_outputs/block_02/crack_seg_local.yaml
Output: /content/drive/MyDrive/vision_unit_02_outputs/block_04


**Training-aligned validation**

In [ ]:
best_model = YOLO(str(BEST_MODEL_PATH))

aligned_metrics = best_model.val(
    data=str(DATA_YAML_PATH),
    split="val",
    imgsz=416,
    batch=16,
    device=0,
    workers=2,

    rect=False,
    nms=True,

    mask_ratio=2,
    overlap_mask=False,

    save_json=False,
    save_txt=False,
    plots=True,

    project=str(OUTPUT_ROOT),
    name="aligned_validation",
    exist_ok=True,
)

In [ ]:
aligned_metric_report = {
    "box_precision": float(
        aligned_metrics.box.mp
    ),
    "box_recall": float(
        aligned_metrics.box.mr
    ),
    "box_map50": float(
        aligned_metrics.box.map50
    ),
    "box_map75": float(
        aligned_metrics.box.map75
    ),
    "box_map50_95": float(
        aligned_metrics.box.map
    ),

    "mask_precision": float(
        aligned_metrics.seg.mp
    ),
    "mask_recall": float(
        aligned_metrics.seg.mr
    ),
    "mask_map50": float(
        aligned_metrics.seg.map50
    ),
    "mask_map75": float(
        aligned_metrics.seg.map75
    ),
    "mask_map50_95": float(
        aligned_metrics.seg.map
    ),

    "speed_ms": {
        key: float(value)
        for key, value
        in aligned_metrics.speed.items()
    },

    "mask_ratio": 2,
    "overlap_mask": False,
    "test_split_used": False,
}

aligned_report_path = (
    OUTPUT_ROOT
    / "aligned_validation_metrics.json"
)

aligned_report_path.write_text(
    json.dumps(
        aligned_metric_report,
        indent=2,
    ),
    encoding="utf-8",
)

print(
    json.dumps(
        aligned_metric_report,
        indent=2,
    )
)

**Confidence behavior and instance matching**

In [ ]:
VALIDATION_IMAGE_DIRECTORY = (
    DATASET_ROOT
    / "images"
    / "val"
)

VALIDATION_LABEL_DIRECTORY = (
    DATASET_ROOT
    / "labels"
    / "val"
)

ANALYSIS_MIN_CONFIDENCE = 0.01
MATCH_MASK_IOU = 0.50
LOCALIZATION_IOU_FLOOR = 0.10